# GRN 2: prepare DREAM5 Network 1

Use the DREAM5 Network 1 expression matrix and its matching gold-standard edge list. Download the official challenge files from the [DREAM5 Zenodo record](https://zenodo.org/records/17854236) or the [Synapse challenge page](https://www.synapse.org/Synapse:syn2787209). Do not mix a gold standard from one network with another network's expression matrix.

Upload the expression TSV and gold-standard TSV when prompted. This notebook attempts to infer common orientations, then prints the parsed dimensions and columns for a manual sanity check. For this lightweight first pass, candidate pairs are class-balanced by sampling negatives; AUROC/AUPR are therefore measured on this sampled candidate set and must not be compared directly with official DREAM5 leaderboard scores.

In [7]:
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import drive
from google.colab import files

In [8]:
drive.mount('/content/drive/', force_remount=True)
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/"
if not os.path.isdir(PROJECT_DIR):
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)

uploaded=files.upload()
print("Uploaded:",list(uploaded))

Mounted at /content/drive/


Saving DREAM5_Net3_Gold Standard.csv to DREAM5_Net3_Gold Standard.csv
Saving DREAM5_Net3_Input Expression.csv to DREAM5_Net3_Input Expression.csv
Uploaded: ['DREAM5_Net3_Gold Standard.csv', 'DREAM5_Net3_Input Expression.csv']


In [9]:
# Select paths by filename. Expected expression file is gene x condition with gene IDs in first column.
expr_path=next((k for k in uploaded if "expression" in k.lower()), None)
gold_path=next((k for k in uploaded if "gold" in k.lower() or "standard" in k.lower()), None)
if expr_path is None or gold_path is None:
    raise ValueError("Upload both an expression TSV and its matching gold-standard TSV; filenames should include 'expression' and 'gold' or 'standard'.")

# Loaded with sep="," since the uploaded files are comma-separated CSVs.
expr_df=pd.read_csv(expr_path,sep=",",header=None,comment="#")
gold_df=pd.read_csv(gold_path,sep=",",header=None,comment="#")
print("Expression preview:"); display(expr_df.iloc[:4,:6])
print("Gold-standard preview:"); display(gold_df.head())
print("Shapes:",expr_df.shape,gold_df.shape)

/tmp/ipykernel_1100/627264674.py:8: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262

Expression preview:


,0,1,2,3,4,5
0,G1,G2,G3,G4,G5,G6
1,7.1151000,9.3293000,9.5997000,6.9998000,7.4955000,8.8540000
2,7.2840000,9.2212000,9.4961000,6.8697000,7.4101000,8.8075000
3,6.6090000,8.6310000,10.2730000,7.0817000,7.5102000,8.3077000


Gold-standard preview:


,0,1,2
0,G92,G2,1
1,G232,G3,1
2,G55,G4,1
3,G56,G4,1
4,G268,G4,1


Shapes: (806, 4511) (152280, 3)


In [10]:
# Infer matrix orientation from dimensions and whether the first row is a header.
def parse_expression(df):
    # If the dataframe has only 1 column, it might have been loaded with the wrong separator.
    # Let's try re-reading it with auto-detected separator.
    if df.shape[1] == 1:
        df = pd.read_csv(expr_path, sep=None, engine='python', header=None, comment="#")

    # Case A: gene IDs in first column; rows are genes, columns are conditions.
    first_col = df.iloc[:,0].astype(str)
    rest = df.iloc[:,1:].apply(pd.to_numeric, errors="coerce")
    first_row_text = pd.to_numeric(df.iloc[0,1:], errors="coerce").isna().mean() > 0.5
    if rest.shape[1] >= 5 and rest.notna().mean().mean() > 0.8 and rest.shape[0] >= rest.shape[1] and not first_row_text:
        return first_col.to_list(), rest.fillna(0).to_numpy(dtype=np.float32)
    # Case B: header contains gene IDs; rows are conditions, columns are genes.
    # Read with header=0/index_col=0, then transpose to genes x conditions if needed.
    df2 = pd.read_csv(expr_path, sep=None, engine='python', header=0, index_col=0, comment="#")
    df2 = df2.apply(pd.to_numeric, errors="coerce")
    if df2.notna().mean().mean() > 0.8:
        if df2.shape[1] > df2.shape[0]:
            return df2.columns.astype(str).to_list(), df2.fillna(0).to_numpy(dtype=np.float32).T
        return df2.index.astype(str).to_list(), df2.fillna(0).to_numpy(dtype=np.float32)
    # Case C: no header, but the first row is not a gene-ID row; transpose based on dimensions.
    if rest.shape[1] >= 5 and rest.notna().mean().mean() > 0.8:
        vals = rest.fillna(0).to_numpy(dtype=np.float32)
        if vals.shape[1] > vals.shape[0]:
            return first_col.to_list(), vals
        return [str(c) for c in df.columns[1:]], vals.T
    raise ValueError("Could not infer expression orientation. Inspect the preview and adapt parsing.")

gene_names, expr = parse_expression(expr_df)


In [11]:
# Gold-standard formats vary. This supports columns [regulator, target, label] and [regulator, target]
# where two-column files are interpreted as positive edges only. Inspect the preview above before trusting it.
g=gold_df.copy()

# Fallback: if loaded incorrectly with 1 column, attempt to split the comma-separated strings
if g.shape[1] == 1 and g.iloc[:,0].astype(str).str.contains(',').any():
    g = g.iloc[:, 0].str.split(',', expand=True)

if g.shape[1] < 2: raise ValueError("Gold-standard file needs at least two columns.")
# Drop obvious header rows and keep string gene names until mapping.
reg_col=g.iloc[:,0].astype(str).str.strip(); tgt_col=g.iloc[:,1].astype(str).str.strip()
if g.shape[1] >= 3:
    raw_label=pd.to_numeric(g.iloc[:,2],errors="coerce")
    valid=raw_label.notna()
    reg_col,tgt_col,raw_label=reg_col[valid],tgt_col[valid],raw_label[valid]
    # Keep labels explicitly coded as 0/1; if format uses signed/weighted labels, stop for review.
    vals=set(raw_label.unique().tolist())
    if not vals.issubset({0,1}):
        raise ValueError(f"Gold-standard third column is not binary (found {sorted(vals)[:10]}). Inspect file format and adapt labels.")
    labels=raw_label.astype(int).to_numpy()
else:
    labels=np.ones(len(g),dtype=int)
gene_to_idx={name:i for i,name in enumerate(gene_names)}
edges=[]
for a,b,yv in zip(reg_col,tgt_col,labels):
    if a in gene_to_idx and b in gene_to_idx and a!=b:
        edges.append((gene_to_idx[a],gene_to_idx[b],int(yv)))
print("Mapped labelled pairs:",len(edges),"| positives:",sum(e[2]==1 for e in edges))
if not edges:
    # Try stripping leading whitespace from gene names in case of formatting variations
    gene_to_idx={name.strip():i for i,name in enumerate(gene_names)}
    edges=[]
    for a,b,yv in zip(reg_col,tgt_col,labels):
        if a.strip() in gene_to_idx and b.strip() in gene_to_idx and a.strip()!=b.strip():
            edges.append((gene_to_idx[a.strip()],gene_to_idx[b.strip()],int(yv)))
    print("Retried with stripped whitespaces -> Mapped labelled pairs:",len(edges),"| positives:",sum(e[2]==1 for e in edges))
if not edges: raise ValueError("No gold-standard IDs matched expression gene IDs. Check gene-name formatting and file orientation.")

Mapped labelled pairs: 152280 | positives: 2066


In [12]:
from grn_experiment_utils import make_pair_sequences
X,y,pair_ids=make_pair_sequences(expr,edges,tf_indices=None,max_conditions=64,
                                 max_pairs=12000,negatives_per_positive=5,seed=42)
out=os.path.join(PROJECT_DIR,"data","dream5_net3_pair_data.npz")
os.makedirs(os.path.dirname(out),exist_ok=True)
np.savez_compressed(out,X=X,y=y,pair_ids=pair_ids,gene_names=np.asarray(gene_names))
print("Saved:",out,"| X:",X.shape,"| positives:",int(y.sum()),"| prevalence:",float(y.mean()))
print("Note: TF-only candidate restriction is not applied unless a TF list is supplied; report this limitation.")

Saved: /content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/data/dream5_net3_pair_data.npz | X: (12000, 64, 2) | positives: 2066 | prevalence: 0.17216666042804718
Note: TF-only candidate restriction is not applied unless a TF list is supplied; report this limitation.


## Data caveats

DREAM5 expression measurements are a compendium, not independent draws from a single homogeneous population. Gene pairs share expression profiles and genes. A random pair split can leak gene-specific structure between train and test. Results here are descriptive and should be accompanied by this caveat. The sampled-negative protocol also means calibration metrics describe the candidate set created here, not the natural network prevalence.